# Thêm Thư Viện

In [82]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [83]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [84]:
df_data_mon = pd.read_csv("./data_mon.csv")
print(df_data_mon)

      ID_Mon                               TenMon      KyHieu TinhTrang
0        120                  Triết học Mác-Lênin  LLCT130105         5
1        121          Kinh tế chính trị Mác-Lênin  LLCT120205         5
2        122            Chủ nghĩa xã hội khoa học  LLCT120405         5
3        123                 Tư tưởng Hồ Chí Minh  LLCT120314        14
4        124       Lịch sử Đảng Cộng sản Việt Nam  LLCT220514        14
...      ...                                  ...         ...       ...
1813    2170                   Thiết kế đồng phục  UNID324652       NaN
1814    2171   Thiết kế thời trang kỹ thuật số 3D  DIFD343952       NaN
1815    2172  Thiết kế thời trang trên Dress Form  FDDF331952       NaN
1816    2173                  Xây dựng phong cách  FAST422452       NaN
1817    2174            Xử lý hình ảnh thời trang  FAPP322552       NaN

[1818 rows x 4 columns]


## Xử lý data

## Thêm dòng không xác định

In [85]:
new_row = pd.DataFrame({
    'ID_Mon': [0],
    'TenMon': ['(Không xác định)'],
})
df_data_mon = pd.concat([df_data_mon, new_row], ignore_index=True) # Thêm vào dataset
df_data_mon = df_data_mon.sort_values(by='ID_Mon', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_data_mon)

      ID_Mon                               TenMon      KyHieu TinhTrang
0          0                     (Không xác định)         NaN       NaN
1        120                  Triết học Mác-Lênin  LLCT130105         5
2        121          Kinh tế chính trị Mác-Lênin  LLCT120205         5
3        122            Chủ nghĩa xã hội khoa học  LLCT120405         5
4        123                 Tư tưởng Hồ Chí Minh  LLCT120314        14
...      ...                                  ...         ...       ...
1814    2170                   Thiết kế đồng phục  UNID324652       NaN
1815    2171   Thiết kế thời trang kỹ thuật số 3D  DIFD343952       NaN
1816    2172  Thiết kế thời trang trên Dress Form  FDDF331952       NaN
1817    2173                  Xây dựng phong cách  FAST422452       NaN
1818    2174            Xử lý hình ảnh thời trang  FAPP322552       NaN

[1819 rows x 4 columns]


## Xử lý cột TinhTrang 

In [86]:
df_data_mon['TinhTrang'] = pd.to_numeric(df_data_mon['TinhTrang'], errors='coerce')
print(df_data_mon)

      ID_Mon                               TenMon      KyHieu  TinhTrang
0          0                     (Không xác định)         NaN        NaN
1        120                  Triết học Mác-Lênin  LLCT130105        5.0
2        121          Kinh tế chính trị Mác-Lênin  LLCT120205        5.0
3        122            Chủ nghĩa xã hội khoa học  LLCT120405        5.0
4        123                 Tư tưởng Hồ Chí Minh  LLCT120314       14.0
...      ...                                  ...         ...        ...
1814    2170                   Thiết kế đồng phục  UNID324652        NaN
1815    2171   Thiết kế thời trang kỹ thuật số 3D  DIFD343952        NaN
1816    2172  Thiết kế thời trang trên Dress Form  FDDF331952        NaN
1817    2173                  Xây dựng phong cách  FAST422452        NaN
1818    2174            Xử lý hình ảnh thời trang  FAPP322552        NaN

[1819 rows x 4 columns]


### Xử lý data rỗng hoặc " "

In [87]:
df_data_mon = df_data_mon.replace(np.nan, None)
df_data_mon = df_data_mon.replace('', None)
print(df_data_mon)

      ID_Mon                               TenMon      KyHieu TinhTrang
0          0                     (Không xác định)        None      None
1        120                  Triết học Mác-Lênin  LLCT130105       5.0
2        121          Kinh tế chính trị Mác-Lênin  LLCT120205       5.0
3        122            Chủ nghĩa xã hội khoa học  LLCT120405       5.0
4        123                 Tư tưởng Hồ Chí Minh  LLCT120314      14.0
...      ...                                  ...         ...       ...
1814    2170                   Thiết kế đồng phục  UNID324652      None
1815    2171   Thiết kế thời trang kỹ thuật số 3D  DIFD343952      None
1816    2172  Thiết kế thời trang trên Dress Form  FDDF331952      None
1817    2173                  Xây dựng phong cách  FAST422452      None
1818    2174            Xử lý hình ảnh thời trang  FAPP322552      None

[1819 rows x 4 columns]


## Load data

### [Nếu cần] Clear bảng

In [88]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Mon"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [89]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Mon (ID_mon, Ten_mon, Ky_hieu, Tinh_trang) 
                VALUES (?, ?, ?, ?)
               """
for index, row in df_data_mon.iterrows():
    values = (row['ID_Mon'], row['TenMon'], row['KyHieu'], row['TinhTrang'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()